## Risk Data to Hexbin
#### Doing all different data types for geographic risk in one notebook.

In [ ]:
import os

os.chdir("")

os.getcwd()

In [1]:
import numpy as np
from shapely.geometry import Point, Polygon
import pandas as pd
import matplotlib.pyplot as plt
import folium
import base64
import geopandas as gpd
import pathlib as Path
import geopandas as gpd
from shapely import union_all
import fiona
import rasterio
from rasterstats import zonal_stats


In [ ]:
#reading the buffered hex bins that was already made-
hex_path = ".gpkg"
#Define for GPD:
gdf_hex = gpd.read_file(hex_path)

# Ensure same crs
projected_crs = "EPSG:3968"
gdf_hex = gdf_hex.to_crs(projected_crs)

In [ ]:
#wildfire is .tif
#heat is .lpkx

#In heat, if "Classify.Class value" = 5,6,7, record number in gdf_hex
#In wildfire, if "UniqueValue.Pixel Value" = 3,4 record number in gdf_hex

#Historical Events shapefile has column "EVENT_TYPE"
#Need each event to be it's own column in gdf_hex, so each unique "EVENT_TYPE" becomes a column
#There are hex bins that have multiple events, and multiple of the same events, need to check for that to be true

### Heat Layer

In [ ]:
print("Hex CRS:", gdf_hex.crs)

with rasterio.open(heat_raster) as src:
    print("Raster CRS:", src.crs)

In [ ]:
#reading the buffered hex bins that was already made-
hex_path = r".gpkg"
#Define for GPD:
gdf_hex = gpd.read_file(hex_path)


heat_raster = r".tif"

with rasterio.open(heat_raster) as src:
    raster_crs = "EPSG:3857"   # force correct CRS

gdf_hex = gdf_hex.to_crs(raster_crs)

#Counting each pixel value per hex
heat_stats = zonal_stats(
    gdf_hex,
    heat_raster,
    categorical=True,
    nodata=0
)

# Count number of extreme heat pixels (values 5,6,7)
heat_stats = zonal_stats(
    gdf_hex,
    heat_raster,
    categorical=True
)

# Compute extreme heat metrics
heat_counts = []
heat_avgs = []

for stat in heat_stats:
    c3 = stat.get(3,0)
    c4 = stat.get(4,0)
    c5 = stat.get(5,0)

    total = c3 + c4 + c5

    heat_counts.append(total)

    if total > 0:
        avg = (3*c3 + 4*c4 + 5*c5) / total
    else:
        avg = 0

    heat_avgs.append(avg)

gdf_hex["heat_count"] = heat_counts
gdf_hex["heat_avg_extreme"] = heat_avgs


# Min-max normalization
min_val = gdf_hex["heat_avg_extreme"].min()
max_val = gdf_hex["heat_avg_extreme"].max()

gdf_hex["Heat_Index"] = (gdf_hex["heat_avg_extreme"] - min_val) / (max_val - min_val)
print(gdf_hex["Heat_Index"].describe())

#name and save where H3 edits are going
gdf_hex.to_file(r'')

### Wildfire Layer

In [ ]:
#reading the buffered hex bins that was already made-
hex_path = r".gpkg"
gdf_hex = gpd.read_file(hex_path)

wildfire_raster = r".tif"

with rasterio.open(wildfire_raster) as src:
    raster_crs = src.crs

gdf_hex = gdf_hex.to_crs(raster_crs)

# Compute zonal stats
wildfire_stats = zonal_stats(
    gdf_hex,
    wildfire_raster,
    categorical=True
)

wildfire_counts = []
wildfire_avg = []

for stat in wildfire_stats:

    if not stat:
        wildfire_counts.append(0)
        wildfire_avg.append(0)
        continue

    c3 = stat.get(3, 0)
    c4 = stat.get(4, 0)

    total = c3 + c4
    wildfire_counts.append(total)

    if total > 0:
        avg = (3*c3 + 4*c4) / total
    else:
        avg = 0

    wildfire_avg.append(avg)

# Add columns
gdf_hex["wildfire_count"] = wildfire_counts
gdf_hex["wildfire_avg"] = wildfire_avg

# Min-max normalization
min_val = gdf_hex["wildfire_avg"].min()
max_val = gdf_hex["wildfire_avg"].max()

gdf_hex["Wildfire_Index"] = (gdf_hex["wildfire_avg"] - min_val) / (max_val - min_val)
print(gdf_hex["Wildfire_Index"].describe())

#name and save where H3 edits are going
gdf_hex.to_file(r'')

## Historical Events

In [ ]:
gdf_hex = gdf_hex.drop(columns=['Flash Flood', 'Flood', 'Funnel Cloud', 'Hail', 'Heavy Rain', 'Lightning', 'Thunderstorm Wind', 'Tornado'])

In [ ]:
#read data and reproject - was a shapefile
historical = gpd.read_file(r'')
historical = historical.to_crs(projected_crs)


In [ ]:
hist_join = gpd.sjoin(gdf_hex, historical, predicate="intersects")

event_counts = (
    hist_join
    .groupby("h3_ID")        
    .size()                  
)

event_counts = event_counts.rename("event_count").to_frame()

gdf_hex = gdf_hex.merge(event_counts, on="h3_ID", how="left")

# Fill NaN (hexes with no events) with 0 and convert to int
gdf_hex["event_count"] = gdf_hex["event_count"].fillna(0).astype(int)

print(gdf_hex.head())

In [ ]:
#name and save where H3 edits are going
gdf_hex.to_file(r'')

In [ ]:
#checking events in our region
# gdf_historical = gpd.read_file(r'')
# historical_risk_list = gdf_historical.columns.tolist()
# pd.Series(historical_risk_list).to_csv("historical_risk.csv", index=False)

In [ ]:
#Adding an Index for historical events
gdf_historical = gpd.read_file(r'')

# Min-max normalization
min_val = gdf_historical["event_count"].min()
max_val = gdf_historical["event_count"].max()

gdf_historical["Event_Index"] = (gdf_historical["event_count"] - min_val) / (max_val - min_val)
print(gdf_historical["Event_Index"].describe())

#overwrite the gdf_historical that was read above 
gdf_historical.to_file(r'')

## Flood Acreage

In [ ]:
#read data and reproject - was a shapefile
flood = gpd.read_file(r'')
flood = flood.to_crs(projected_crs)

#Flood
flood_hex = gpd.overlay(gdf_hex, flood, how="intersection")
flood_hex["flood_area_m2"] = flood_hex.geometry.area
flood_hex["flood_acres"] = flood_hex["flood_area_m2"] / 4046.85642

flood_summary = flood_hex.groupby("h3_ID")["flood_acres"].sum()
gdf_hex["flood_acres"] = gdf_hex["h3_ID"].map(flood_summary).fillna(0)

# Min-max normalization
min_val = gdf_hex["flood_acres"].min()
max_val = gdf_hex["flood_acres"].max()

gdf_hex["Flood_Index"] = (gdf_hex["flood_acres"] - min_val) / (max_val - min_val)
print(gdf_hex["Flood_Index"].describe())


#name and save where H3 edits are going
gdf_hex.to_file(r'')

## Sea Level Rise Acreage

In [ ]:
# read data and reproject - was a shapefile
sea = gpd.read_file(r'')
sea = sea.to_crs(projected_crs)

#sea
sea_hex = gpd.overlay(gdf_hex, sea, how="intersection")
sea_hex["area_m2"] = sea_hex.geometry.area
sea_hex["sea_acres"] = sea_hex["area_m2"] / 4046.85642

sea_summary = sea_hex.groupby("h3_ID")["sea_acres"].sum()
gdf_hex["sea_acres"] = gdf_hex["h3_ID"].map(sea_summary).fillna(0)


# Min-max normalization
min_val = gdf_hex["sea_acres"].min()
max_val = gdf_hex["sea_acres"].max()

gdf_hex["Sea_Index"] = (gdf_hex["sea_acres"] - min_val) / (max_val - min_val)
print(gdf_hex["Sea_Index"].describe())

#name and save where H3 edits are going
gdf_hex.to_file(r'')

## Dam Inundation Acreage

In [ ]:
#read data and project - was a shapefile
dam = gpd.read_file(r'')

dam = dam.to_crs(projected_crs)

#dam
dam_hex = gpd.overlay(gdf_hex, dam, how="intersection")
dam_hex["area_m2"] = dam_hex.geometry.area
dam_hex["dam_acres"] = dam_hex["area_m2"] / 4046.85642

dam_summary = dam_hex.groupby("h3_ID")["dam_acres"].sum()
gdf_hex["dam_acres"] = gdf_hex["h3_ID"].map(dam_summary).fillna(0)

# Min-max normalization
min_val = gdf_hex["dam_acres"].min()
max_val = gdf_hex["dam_acres"].max()

gdf_hex["Dam_Index"] = (gdf_hex["dam_acres"] - min_val) / (max_val - min_val)
print(gdf_hex["Dam_Index"].describe())

#name and save where H3 edits are going
gdf_hex.to_file(r'')

## Abandoned Mines Problem Areas

In [ ]:
#read data and reproject - was a shapefile
areas = gpd.read_file(r'')
areas = areas.to_crs(projected_crs)

#sea
pa_hex = gpd.overlay(gdf_hex, areas, how="intersection")
pa_hex["area_m2"] = pa_hex.geometry.area
pa_hex["pa_acres"] = pa_hex["area_m2"] / 4046.85642

pa_summary = pa_hex.groupby("h3_ID")["pa_acres"].sum()
gdf_hex["pa_acres"] = gdf_hex["h3_ID"].map(pa_summary).fillna(0)

# Min-max normalization
min_val = gdf_hex["pa_acres"].min()
max_val = gdf_hex["pa_acres"].max()

gdf_hex["PA_Index"] = (gdf_hex["pa_acres"] - min_val) / (max_val - min_val)
print(gdf_hex["PA_Index"].describe())

#name and save where H3 edits are going
gdf_hex.to_file(r'')